Load the pandas library first

In [29]:
import pandas as pd
import numpy as np
from collections import defaultdict
from difflib import get_close_matches
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import coo_matrix  

Then load the datasets 

In [30]:
user_history = pd.read_csv('C:/Users/Lwin/Python Deep Learning/User Listening History.csv')
music_info = pd.read_csv('C:/Users/Lwin/Python Deep Learning/Music Info.csv')

encode user and track id

In [31]:
user_encoder = LabelEncoder()
track_encoder = LabelEncoder()
user_history['user_index'] = user_encoder.fit_transform(user_history['user_id'])
user_history['track_index'] = track_encoder.fit_transform(user_history['track_id'])

create sparse item matrix

In [32]:
rows = user_history['user_index']
cols = user_history['track_index']
data = user_history['playcount'].astype(np.float32)
num_users = rows.nunique()
num_items = cols.nunique()
user_item_matrix = coo_matrix((data, (rows, cols)), shape=(num_users, num_items)).tocsr()

apply truncated SVD

In [33]:
svd = TruncatedSVD(n_components=20, random_state=42)
svd_matrix = svd.fit_transform(user_item_matrix)
predicted_ratings = svd_matrix @ svd.components_

convert index to id to map them

In [34]:
track_id_to_info = music_info.set_index('track_id')[['name', 'artist']].to_dict('index')
user_index_to_id = dict(enumerate(user_encoder.classes_))
track_index_to_id = dict(enumerate(track_encoder.classes_))

recommend based on track id then compare for similarity then suggest song that havent exist in the album of the user input id

In [37]:
import pandas as pd
import numpy as np
from collections import defaultdict
from difflib import get_close_matches
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import coo_matrix

# Load data
user_history = pd.read_csv('C:/Users/Lwin/Python Deep Learning/User Listening History.csv')
music_info = pd.read_csv('C:/Users/Lwin/Python Deep Learning/Music Info.csv')

# Encode users and tracks
user_encoder = LabelEncoder()
track_encoder = LabelEncoder()
user_history['user_index'] = user_encoder.fit_transform(user_history['user_id'])
user_history['track_index'] = track_encoder.fit_transform(user_history['track_id'])

# Create user-item matrix
rows = user_history['user_index']
cols = user_history['track_index']
data = user_history['playcount'].astype(np.float32)
num_users = rows.nunique()
num_items = cols.nunique()
user_item_matrix = coo_matrix((data, (rows, cols)), shape=(num_users, num_items)).tocsr()

# Apply SVD for dimensionality reduction
svd = TruncatedSVD(n_components=20, random_state=42)
svd_matrix = svd.fit_transform(user_item_matrix)
predicted_ratings = svd_matrix @ svd.components_

# Create lookup dictionaries
track_id_to_info = music_info.set_index('track_id')[['name', 'artist']].to_dict('index')
user_index_to_id = dict(enumerate(user_encoder.classes_))
track_index_to_id = dict(enumerate(track_encoder.classes_))


def recommend_by_user_id(user_id_input, top_n=5):
    if user_id_input not in user_encoder.classes_:
        suggestions = get_close_matches(user_id_input, user_encoder.classes_, n=3)
        return f"User ID not found. Did you mean: {', '.join(suggestions)}?"

    user_index = user_encoder.transform([user_id_input])[0]
    target_vector = svd_matrix[user_index]
    
    # Calculate cosine similarity with ALL users
    similarities = cosine_similarity([target_vector], svd_matrix)[0]

    # Get top 5 similar users (excluding self)
    top_user_indices = np.argsort(-similarities)[1:6]
    
    print(f"\nInput User ID: {user_id_input}")
    print("Top 5 similar users and their similarity scores:")
    user_similarities = {}
    for i, user_idx in enumerate(top_user_indices, 1):
        sim_id = user_index_to_id[user_idx]
        sim_score = similarities[user_idx]
        user_similarities[user_idx] = sim_score
        print(f" {i}. {sim_id} (Similarity: {sim_score:.4f})")

    print(f"\nAnalyzing songs from these {len(top_user_indices)} similar users...")
    
    # Track which users listened to which songs
    track_to_user_similarities = defaultdict(list)
    
    for sim_idx in top_user_indices:
        uid = user_index_to_id[sim_idx]
        sim_score = similarities[sim_idx]
        user_tracks = user_history[user_history['user_index'] == sim_idx]['track_index'].unique()
        
        for tidx in user_tracks:
            track_to_user_similarities[tidx].append({
                'user_id': uid,
                'user_index': sim_idx,
                'similarity': sim_score
            })

    # Get songs user has already listened to
    listened = set(user_history[user_history['user_index'] == user_index]['track_index'].unique())
    
    # Score tracks based on weighted average of similar users' preferences
    track_scores = {}
    for track_idx, user_list in track_to_user_similarities.items():
        if track_idx not in listened:  # Only recommend unheard songs
            # Weight by similarity scores
            weighted_score = sum(user_info['similarity'] for user_info in user_list)
            track_scores[track_idx] = {
                'score': weighted_score,
                'contributing_users': user_list
            }
    
    # Sort by weighted score and get top N
    top_tracks = sorted(track_scores.items(), key=lambda x: x[1]['score'], reverse=True)[:top_n]
    
    recommendations = []
    for track_idx, track_data in top_tracks:
        track_id = track_index_to_id[track_idx]
        info = track_id_to_info.get(track_id, {'name': 'Unknown', 'artist': 'Unknown'})
        
        # Format contributing users with their similarities
        contributing_users = []
        for user_info in track_data['contributing_users']:
            contributing_users.append(f"{user_info['user_id']} ({user_info['similarity']:.4f})")
        
        recommendations.append({
            'track_id': track_id,
            'name': info['name'],
            'artist': info['artist'],
            'weighted_score': track_data['score'],
            'num_similar_users': len(track_data['contributing_users']),
            'contributing_users_with_similarity': '; '.join(contributing_users)
        })
    
    return pd.DataFrame(recommendations)


# Main execution
user_input = input("Enter a User ID to get recommendations: ")
results = recommend_by_user_id(user_input, top_n=5)

# Display results
if isinstance(results, pd.DataFrame):
    pd.set_option('display.max_colwidth', None)
    print(f"\nTop {len(results)} song recommendations:")
    print("="*80)
    
    for idx, row in results.iterrows():
        print(f"\n{idx+1}. '{row['name']}' by {row['artist']}")
        print(f"   Track ID: {row['track_id']}")
        print(f"   Weighted Score: {row['weighted_score']:.4f}")
        print(f"   Recommended by {row['num_similar_users']} similar user(s):")
        
        # Split and format the contributing users
        users = row['contributing_users_with_similarity'].split('; ')
        for i, user in enumerate(users, 1):
            print(f"     {i}. {user}")
    
    print("\n" + "="*80)
    print("DataFrame Summary:")
    print(results[['name', 'artist', 'weighted_score', 'num_similar_users']].to_string(index=False))
    
else:
    print(results)

Enter a User ID to get recommendations:  85C1F87FEA955D09B4BEC2E36AEE110927AEDF9A



Input User ID: 85C1F87FEA955D09B4BEC2E36AEE110927AEDF9A
Top 5 similar users and their similarity scores:
 1. BF458A99EA1C7DAF787DC32E2143B0A1452751F5 (Similarity: 0.9993)
 2. 14A4F29658E87827AB400FA5AAA947FB4A9B0672 (Similarity: 0.9674)
 3. 8FCBE0C749DA0559932D34697BBE20BACC70E869 (Similarity: 0.9598)
 4. 94F7C69CC2BECC84EEAEA48C7CB5B8455E5FB5BF (Similarity: 0.9595)
 5. 4DF273E8093BEBF72996E7626242B5E9BD64ECD9 (Similarity: 0.9591)

Analyzing songs from these 5 similar users...

Top 5 song recommendations:

1. 'Red Dirt Road' by BROOKS & DUNN
   Track ID: TROXWBT128F9305E67
   Weighted Score: 1.9271
   Recommended by 2 similar user(s):
     1. 14A4F29658E87827AB400FA5AAA947FB4A9B0672 (0.9674)
     2. 8FCBE0C749DA0559932D34697BBE20BACC70E869 (0.9598)

2. 'Dead On Arrival' by EVOL INTENT
   Track ID: TRJPKFT12903CD8434
   Weighted Score: 0.9993
   Recommended by 1 similar user(s):
     1. BF458A99EA1C7DAF787DC32E2143B0A1452751F5 (0.9993)

3. 'Buddy Holly' by WEEZER
   Track ID: TRWQINA12

Recommend top N songs for track selected